## Preprocessing

In [29]:
''' preserve word order, keep ! and ? as tokens, and use a minimum document frequency of 2 for the vocabulary'''

import re
from collections import Counter
import numpy as np

train_texts, train_labels = load_text('train_data.csv')
val_texts, val_labels = load_text('val_data.csv')
test_texts, test_labels = load_text('test_data.csv')


token_pattern = re.compile(r"[\w!?]+") #keep the ! and ? just like before

def tokenize(text):
    return token_pattern.findall(text)

def build_vocab(texts, min_df=2):
    token_counts = Counter()
    for text in texts:
        tokens = tokenize(text)
        token_counts.update(tokens)
    return {token for token, count in token_counts.items() if count >= min_df}  #return tokens that appear at least min_df times matching other model

vocab = build_vocab(train_texts, min_df=2)  

word2idx = {'<PAD>': 0, '<UNK>': 1}  # reserve index 0 for padding and 1 for unknown tokens
start_idx = 2
for word in vocab:
    word2idx[word] = start_idx
    start_idx += 1              #assign index to each word in vocab starting with 2

vocab_size = len(word2idx)



def tokens_index(tokens, word2idx):
    return [word2idx.get(token, word2idx['<UNK>']) for token in tokens]  #look up the index of each token in word2idx, if not found return index of <UNK>


train_tokenized = [tokens_index(tokenize(text), word2idx) for text in train_texts]
val_tokenized = [tokens_index(tokenize(text), word2idx) for text in val_texts]
test_tokenized = [tokens_index(tokenize(text), word2idx) for text in test_texts]

maxlen = max(len(tokens) for tokens in train_tokenized)  #find the maximum length of tokens in training data, are similar lengths already 

def pad_sequence(ids, maxlen, pad_id=0):
    if len(ids) >= maxlen:
        return ids[:maxlen]
    return ids + [pad_id] * (maxlen - len(ids))

train_padded = np.array([pad_sequence(ids, maxlen) for ids in train_tokenized])
train_mask = (train_padded !=0).astype(int) #create an array indicating 1 for real token vs padding 0 will use later




NameError: name 'load_text' is not defined

In [30]:
import numpy as np

def init_embeddings(vocab_size, embedding_dims=50):
    embedding_matrix = np.random.randn(vocab_size, embedding_dims) * 0.01 #can vary this later
    return embedding_matrix

def embedding_forward(token_ids, embedding_matrix):
    embedded = embedding_matrix[token_ids] #match the token ids to our embedding matrix
    return embedded

def embedding_backward(d_embedded, token_ids, vocab_size, embedding_dims):  
    d_embedding_matrix = np.zeros((vocab_size, embedding_dims)) 
    np.add.at(d_embedding_matrix, token_ids, d_embedded) #allows us to process multiple identical ids in the same document wihtout overwriting
    return d_embedding_matrix


In [ ]:
# quick check
token_ids = [[2,2]]
d_embedded = [[1,1]]

embedding_backward(d_embedded, token_ids, vocab_size=1000, embedding_dims=2) #simple check that gradients correctly sum without overwriting

array([[0., 0.],
       [0., 0.],
       [2., 2.],
       ...,
       [0., 0.],
       [0., 0.],
       [0., 0.]], shape=(1000, 2))

In [31]:
def init_weights(d_k, embedding_dims=50):
    wq =  np.random.randn(embedding_dims, d_k) * 0.01
    wk = np.random.randn(embedding_dims, d_k) * 0.01
    wv = np.random.randn(embedding_dims, d_k) * 0.01
    return wq, wk, wv

def softmax(x, axis=-1):
    x_stable = x - np.max(x, axis=axis, keepdims=True) #for numerical stability precent overflow
    exp_x = np.exp(x_stable)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def forward_pass_att(embedded, mask, wq, wk, wv, d_k):
    Q = embedded @ wq
    K = embedded @ wk
    V = embedded @ wv
    scores = Q @ K.transpose(0, 2, 1) 
    scores = scores / np.sqrt(d_k)
    mask_expanded = mask[:, None, :] #reshape mask to match score
    scores = np.where(mask_expanded == 0, -1e9, scores) #exp needs to be close to zero so can't use 0 as exp(0)=1
    attn_weights = softmax(scores, axis=-1)
    attention_output = attn_weights @ V 
    return attention_output, attn_weights, Q, K, V




## Checks

In [32]:
# sanity check

embedded = np.array([[[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]])  # shape: (1, 3, 2)
d_k = 2

# use identity matrices so Q, K, V = embedded
wq = np.eye(2)
wk = np.eye(2)
wv = np.eye(2)

# test 1: no padding
mask_no_pad = np.array([[1, 1, 1]])
out, weights, Q, K, V = forward_pass_att(
    embedded, mask_no_pad, wq, wk, wv, d_k
)

assert np.allclose(Q, embedded), "Q should match input when Wq is identity"
print("Unmasked weights shape:", weights.shape)
print("Unmasked weights:\n", np.round(weights, 4))
print("Unmasked output:\n", np.round(out, 4))

# test 2: mask out token 2 (pad token)
mask_with_pad = np.array([[1, 1, 0]])
out, weights, Q, K, V = forward_pass_att(
    embedded, mask_with_pad, wq, wk, wv, d_k
)

print("\nMasked weights (col 2 should be ~0):\n", np.round(weights, 4))

# check that the padded key position gets near-zero attention
assert np.all(weights[:, :, 2] < 1e-5), "Masked token received attention!"
print("\nAll checks passed.")

Unmasked weights shape: (1, 3, 3)
Unmasked weights:
 [[[0.4011 0.1978 0.4011]
  [0.1978 0.4011 0.4011]
  [0.2483 0.2483 0.5035]]]
Unmasked output:
 [[[0.8022 0.5989]
  [0.5989 0.8022]
  [0.7517 0.7517]]]

Masked weights (col 2 should be ~0):
 [[[0.6698 0.3302 0.    ]
  [0.3302 0.6698 0.    ]
  [0.5    0.5    0.    ]]]

All checks passed.


## Pooling and backprop for attention

In [33]:
import numpy as np


def masked_pool(attention_output, mask):
    # zero out pad tokens
    mask_expanded = mask[:, :, None]
    masked_output = attention_output * mask_expanded
    
    # average across valid tokens
    summed_output = masked_output.sum(axis=1)
    counts = mask.sum(axis=1, keepdims=True)
    pooled = summed_output / counts

    return pooled, counts


def back_pool(d_pool, mask, counts):
    d_sum = d_pool / counts
    d_masked = d_sum[:, None, :]
    d_attention_output = d_masked * mask[:, :, None]

    return d_attention_output


def attention_backward(d_attention_output, Q, K, V, attn_weights, mask, embedded, W_Q, W_K, W_V, d_k):
    # grad for attention output mult
    d_attn_weights = d_attention_output @ V.transpose(0, 2, 1)   
    d_V = attn_weights.transpose(0, 2, 1) @ d_attention_output   
    
    # softmax grad + apply mask
    sum_term = (d_attn_weights * attn_weights).sum(axis=-1, keepdims=True)
    d_scores = attn_weights * (d_attn_weights - sum_term)
    d_scores_masked = d_scores * mask[:, None, :]
    
    # scale factor grad
    d_raw_scores = d_scores_masked / np.sqrt(d_k)
    
    # grad w.r.t Q and K
    d_Q = d_raw_scores @ K                             
    d_K = d_raw_scores.transpose(0, 2, 1) @ Q 
    
    # weight grads
    d_W_Q = np.einsum('bld,ble->de', embedded, d_Q)      
    d_W_K = np.einsum('bld,ble->de', embedded, d_K)
    d_W_V = np.einsum('bld,ble->de', embedded, d_V)
    
    # input embedding grads combined from all branches
    d_emb_Q = d_Q @ W_Q.T
    d_emb_K = d_K @ W_K.T
    d_emb_V = d_V @ W_V.T 
    d_embedded = d_emb_Q + d_emb_K + d_emb_V   

    return d_embedded, d_W_Q, d_W_K, d_W_V




## Checks

In [35]:
# mock inputs with tiny dimensions
np.random.seed(42)
b, l, d, d_k = 1, 2, 2, 2

embedded = np.random.randn(b, l, d)
W_Q = np.random.randn(d, d_k)
W_K = np.random.randn(d, d_k)
W_V = np.random.randn(d, d_k)

# forward Pass
Q = embedded @ W_Q
K = embedded @ W_K
V = embedded @ W_V
scores = (Q @ K.transpose(0, 2, 1)) / np.sqrt(d_k)
e_x = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
attn_weights = e_x / e_x.sum(axis=-1, keepdims=True)
mask = np.ones((b, l))

d_out = np.ones((b, l, d_k))  # incoming gradient

# analytical Gradient
d_attn = d_out @ V.transpose(0, 2, 1)
d_V = attn_weights.transpose(0, 2, 1) @ d_out
d_scores = attn_weights * (d_attn - (d_attn * attn_weights).sum(axis=-1, keepdims=True))
d_raw = d_scores / np.sqrt(d_k)

d_Q = d_raw @ K
d_K = d_raw.transpose(0, 2, 1) @ Q

d_embedded = (d_Q @ W_Q.T) + (d_K @ W_K.T) + (d_V @ W_V.T)
d_W_Q = np.einsum('bld,ble->de', embedded, d_Q)

# simple numerical check for 1 element in W_Q[0, 0]
eps = 1e-5

def compute_loss():
    Q_temp = embedded @ W_Q
    scores_temp = (Q_temp @ K.transpose(0, 2, 1)) / np.sqrt(d_k)
    exp_t = np.exp(scores_temp - np.max(scores_temp, axis=-1, keepdims=True))
    weights_t = exp_t / exp_t.sum(axis=-1, keepdims=True)
    out_t = weights_t @ V
    return np.sum(out_t * d_out) 

# numerical derivative 
W_Q[0, 0] += eps
loss_high = compute_loss()

W_Q[0, 0] -= 2 * eps
loss_low = compute_loss()

W_Q[0, 0] += eps  # reset original value

num_grad = (loss_high - loss_low) / (2 * eps)
analytic_grad = d_W_Q[0, 0]

print(f"Analytical Gradient: {analytic_grad:.6f}")
print(f"Numerical Gradient: {num_grad:.6f}")


Analytical Gradient: 0.477056
Numerical Gradient: 0.477056
